<a href="https://colab.research.google.com/github/AliMahmoud67/FlyRank_Starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliMahmoud67/FlyRank_Starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [19]:
import os
import duckdb
from google.colab import userdata

# Get Hugging Face token from Colab Secret
HF_TOKEN = userdata.get("HF_TOKEN")

# Create DuckDB connection
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Authenticate with Hugging Face
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Connected to FlyRank warehouse.")

Connected to FlyRank warehouse.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


Prioritize content that is old and has weak click-through performance relative to its search position.

The rule uses two signals:

- **Staleness:** `content_age_days`
- **CTR vs position:** February CTR together with `gsc_avg_position`

Reason codes:
- `stale_and_weak_ctr`
- `stale_only`
- `weak_ctr_only`
- `no_signal`

In [20]:
import pandas as pd
import numpy as np

FEB = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet"
DIM = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

feature_vector = con.execute(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        SUM(f.gsc_impressions) AS gsc_impressions,
        SUM(f.gsc_clicks) AS gsc_clicks,

        SUM(f.gsc_sum_position)
            / NULLIF(SUM(f.gsc_impressions), 0) AS gsc_avg_position,

        d.word_count,

        DATE_DIFF(
            'day',
            d.content_created_date,
            DATE '2026-02-28'
        ) AS content_age_days

    FROM read_parquet('{FEB}') f

    JOIN read_parquet('{DIM}') d
        ON f.content_hash_id = d.content_hash_id

    WHERE f.gsc_data_available IS TRUE
      AND d.content_created_date <= DATE '2026-02-28'

    GROUP BY
        f.client_hash_id,
        f.content_hash_id,
        d.word_count,
        d.content_created_date
""").df()

print("Feature rows:", len(feature_vector))
print("Columns:", feature_vector.columns.tolist())

feature_vector.head()

Feature rows: 153559
Columns: ['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'word_count', 'content_age_days']


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,word_count,content_age_days
0,client_e547b89c05043229,content_7995404695ee1ffd,1012.0,3.0,28.886364,2828,226
1,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,12.448161,2613,226
2,client_e547b89c05043229,content_ae16a6b9cf64c80a,861.0,0.0,8.182346,2928,226
3,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.316508,2992,226
4,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,9.966926,2225,226


In [21]:
MAR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

march = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS march_ctr
    FROM read_parquet('{MAR}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= 100
""").df()

check = feature_vector.merge(
    march,
    on=["client_hash_id", "content_hash_id"]
)

print("Rows we can compare:", len(check))

Rows we can compare: 86560


In [22]:
check["age_group"] = pd.cut(
    check["content_age_days"],
    [-1, 90, 180, 9999],
    labels=["0-90 days", "91-180 days", "181+ days"]
)

print(
    check.groupby("age_group", observed=False)
    .agg(
        n=("content_hash_id", "count"),
        march_ctr=("march_ctr", "mean")
    )
)

                 n  march_ctr
age_group                    
0-90 days    25531   0.002588
91-180 days  16853   0.002328
181+ days    44176   0.002316


**Staleness verdict: MIXED**

Older content had slightly lower March CTR, but the difference between the 91–180 day and 181+ day groups was small. Staleness is therefore a useful signal to consider, but it is not a strong standalone signal.

In [23]:
# February CTR
check["feb_ctr"] = (
    check["gsc_clicks"] /
    check["gsc_impressions"].replace(0, np.nan)
)

# Group pages by their search position
check["position_group"] = pd.cut(
    check["gsc_avg_position"],
    [0, 3, 10, 20],
    labels=["1-3", "4-10", "11-20"]
)

# Rank CTR within each position group
check["ctr_rank"] = (
    check[check["position_group"].notna()]
    .groupby("position_group", observed=True)["feb_ctr"]
    .rank(pct=True)
)

# Bottom 25% CTR for its position group
check["weak_ctr_vs_position"] = check["ctr_rank"] <= 0.25

print(
    check[check["position_group"].notna()]
    .groupby("weak_ctr_vs_position")
    .agg(
        n=("content_hash_id", "count"),
        march_ctr=("march_ctr", "mean")
    )
)

                          n  march_ctr
weak_ctr_vs_position                  
False                 53685   0.002997
True                  19745   0.001646


**CTR-vs-position verdict: CONFIRMED**

Pages with weak February CTR compared with other pages at similar search positions had lower March CTR. This supports using weak CTR relative to position as a baseline opportunity signal.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [24]:
# Build the baseline score

baseline = feature_vector.copy()

# February CTR
baseline["feb_ctr"] = (
    baseline["gsc_clicks"] /
    baseline["gsc_impressions"].replace(0, np.nan)
)

# Signal 1: stale content
baseline["stale"] = baseline["content_age_days"] >= 180

# Signal 2: weak CTR for its position
baseline["position_group"] = pd.cut(
    baseline["gsc_avg_position"],
    [0, 3, 10, 20],
    labels=["1-3", "4-10", "11-20"]
)

baseline["ctr_rank"] = (
    baseline[baseline["position_group"].notna()]
    .groupby("position_group", observed=True)["feb_ctr"]
    .rank(method="min", pct=True)
)

baseline["weak_ctr"] = baseline["ctr_rank"] <= 0.25

# Score
baseline["score"] = (
    baseline["stale"].astype(int) +
    baseline["weak_ctr"].fillna(False).astype(int)
)

# Action
baseline["action"] = np.where(
    baseline["score"] == 2,
    "Refresh content",
    np.where(
        baseline["score"] == 1,
        "Review content",
        "No action"
    )
)

# Reason code
baseline["reason_code"] = np.select(
    [
        baseline["stale"] & baseline["weak_ctr"],
        baseline["stale"],
        baseline["weak_ctr"]
    ],
    [
        "stale_and_weak_ctr",
        "stale",
        "weak_ctr"
    ],
    default="no_signal"
)

# Rank
baseline = baseline.sort_values(
    ["score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

baseline["rank"] = range(1, len(baseline) + 1)

queue = baseline[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "action",
        "reason_code"
    ]
]

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written.")
print("Rows:", len(queue))

print("\nScore distribution:")
print(baseline["score"].value_counts().sort_index())

print("\nReason code distribution:")
print(baseline["reason_code"].value_counts())

queue.head(10)

CSV written.
Rows: 153559

Score distribution:
score
0    33267
1    84626
2    35666
Name: count, dtype: int64

Reason code distribution:
reason_code
stale                 44810
weak_ctr              39816
stale_and_weak_ctr    35666
no_signal             33267
Name: count, dtype: int64


,rank,client_hash_id,content_hash_id,score,action,reason_code
0,1,client_73cda7b4e4f265ea,content_fec55986a1868d62,2,Refresh content,stale_and_weak_ctr
1,2,client_73cda7b4e4f265ea,content_c9f840183215651b,2,Refresh content,stale_and_weak_ctr
2,3,client_23a62021009f63c4,content_2f09787bdf392b16,2,Refresh content,stale_and_weak_ctr
3,4,client_73cda7b4e4f265ea,content_1cb7263083e97ba1,2,Refresh content,stale_and_weak_ctr
4,5,client_73cda7b4e4f265ea,content_d16bbebfbb3c8fda,2,Refresh content,stale_and_weak_ctr
5,6,client_23a62021009f63c4,content_66d3e7a515e4ec68,2,Refresh content,stale_and_weak_ctr
6,7,client_73cda7b4e4f265ea,content_4b3ab5ebb70090f1,2,Refresh content,stale_and_weak_ctr
7,8,client_23a62021009f63c4,content_dcc8191464a7e5b0,2,Refresh content,stale_and_weak_ctr
8,9,client_73cda7b4e4f265ea,content_c5fca21576ab6a02,2,Refresh content,stale_and_weak_ctr
9,10,client_23a62021009f63c4,content_7df93ae01f4a6a21,2,Refresh content,stale_and_weak_ctr


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [27]:
# Review the top 20 ranked items

top20 = baseline.head(20).copy()

top20["confidence_note"] = np.select(
    [
        top20["reason_code"] == "stale_and_weak_ctr",
        top20["reason_code"] == "weak_ctr",
        top20["reason_code"] == "stale"
    ],
    [
        "Medium confidence: both signals are present.",
        "Medium confidence: this signal was confirmed in validation.",
        "Low confidence: staleness showed only a mixed relationship."
    ],
    default="Low confidence."
)

top20["what_would_make_it_wrong"] = np.select(
    [
        top20["reason_code"] == "stale_and_weak_ctr",
        top20["reason_code"] == "weak_ctr",
        top20["reason_code"] == "stale"
    ],
    [
        "The page may be old but still useful and up to date, or low CTR may have another cause.",
        "Low CTR may be caused by SERP features, branding, or the search result snippet rather than the content.",
        "The content may be old but still accurate and useful."
    ],
    default="The signals may not reflect a real refresh opportunity."
)

top20[
    [
        "rank",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

,rank,content_hash_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_fec55986a1868d62,Refresh content,stale_and_weak_ctr,Medium confidence: both signals are present.,The page may be old but still useful and up to...
1,2,content_c9f840183215651b,Refresh content,stale_and_weak_ctr,Medium confidence: both signals are present.,The page may be old but still useful and up to...
2,3,content_2f09787bdf392b16,Refresh content,stale_and_weak_ctr,Medium confidence: both signals are present.,The page may be old but still useful and up to...
3,4,content_1cb7263083e97ba1,Refresh content,stale_and_weak_ctr,Medium confidence: both signals are present.,The page may be old but still useful and up to...
4,5,content_d16bbebfbb3c8fda,Refresh content,stale_and_weak_ctr,Medium confidence: both signals are present.,The page may be old but still useful and up to...
5,6,content_66d3e7a515e4ec68,Refresh content,stale_and_weak_ctr,Medium confidence: both signals are present.,The page may be old but still useful and up to...
6,7,content_4b3ab5ebb70090f1,Refresh content,stale_and_weak_ctr,Medium confidence: both signals are present.,The page may be old but still useful and up to...
7,8,content_dcc8191464a7e5b0,Refresh content,stale_and_weak_ctr,Medium confidence: both signals are present.,The page may be old but still useful and up to...
8,9,content_c5fca21576ab6a02,Refresh content,stale_and_weak_ctr,Medium confidence: both signals are present.,The page may be old but still useful and up to...
9,10,content_7df93ae01f4a6a21,Refresh content,stale_and_weak_ctr,Medium confidence: both signals are present.,The page may be old but still useful and up to...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The top picks are driven by the stale and weak-CTR signals. A pick could still be wrong if the content is old but still useful and up to date, or if low CTR is caused by the search result presentation rather than the content itself.

The baseline score uses only February features. No March outcome fields or label-derived fields are used in the ranking score.

In [26]:
# Check the top 20 for possible weak picks

top20 = baseline.head(20)

print("Top 20 score:", top20["score"].unique())
print("Top 20 reason codes:")
print(top20["reason_code"].value_counts())

print("\nFeatures used for the baseline:")
print([
    "content_age_days",
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position"
])

print("\nNo March outcome fields are used in the baseline score.")
print("No trend_direction, trend_pct, or is_declining_label fields are used.")

Top 20 score: [2]
Top 20 reason codes:
reason_code
stale_and_weak_ctr    20
Name: count, dtype: int64

Features used for the baseline:
['content_age_days', 'gsc_clicks', 'gsc_impressions', 'gsc_avg_position']

No March outcome fields are used in the baseline score.
No trend_direction, trend_pct, or is_declining_label fields are used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.